# Protein complexes with optional paired MSA

Predict a barnase–barstar heterodimer, optionally supply per-chain A3M alignments, and specify two copies of one chain.

Tested with **Boltz2 NIM 1.9.0 / client 0.6.0**.
From the repository root, install `python -m pip install -e ".[notebooks]"`
and start `jupyter lab examples/notebooks` in that environment.
Use a [ready NIM server](../../docs/deployment.md); `BOLTZ2_NIM_URL`
defaults to `http://localhost:8000`. Run cells in order.

[Setup and output files](../README.md#notebooks) ·
[NIM release notes](https://docs.nvidia.com/nim/bionemo/boltz2/1.9.0/release-notes.html)

Copyright (c) 2025-2026 NVIDIA Corporation.

In [ ]:
import os
from pathlib import Path
import boltz2_client
from boltz2_client import Boltz2Client, Polymer, PredictionRequest

client = Boltz2Client(
    base_url=os.getenv("BOLTZ2_NIM_URL", "http://localhost:8000"), timeout=1200,
)
metadata = await client.get_service_metadata()
print(f"Client {boltz2_client.__version__}; connected NIM {metadata.version} (tested: 1.9.0)")
output_dir = Path("notebook_outputs") / "protein_complexes"
output_dir.mkdir(parents=True, exist_ok=True)

## Define the chains

Chain A is mature barnase ([UniProt P00648](https://www.uniprot.org/uniprotkb/P00648/entry),
residues 48–157); chain B is mature barstar
([P11540](https://www.uniprot.org/uniprotkb/P11540/entry), residues 2–90).
Each polymer has a unique chain ID.

In [ ]:
chain_sequences = {
    "A": "AQVINTFDGVADYLQTYHKLPDNYITKSEAQALGWVASKGNLADVAPGKSIGGDIFSNREGKLPGKSGRTWREADINYTSGFRNSDRILYSSDWLIYKTTDHYQTFTKIR",
    "B": "KKAVINGEQIRSISDLHQTLKKELALPEYYGENLDALWDCLTGWVEYPLVLEWRQFEQSKQLTENGAESVLQVFREAKAEGCDITIILS",
}

## Optional: pair existing A3M files

Set both `MULTIMER_A3M_A` and `MULTIMER_A3M_B` to per-chain A3M files before
launching Jupyter. Their first sequences must match the chains above; edit
`chain_sequences` when using another target. Greedy pairing matches homologs
using their taxonomy/UniRef headers and creates Boltz2 CSV records per chain.

Without files, prediction runs without an MSA. This cell performs no homolog
search or database download. See [pairing options](../../docs/a3m_to_multimer_msa.md).

In [ ]:
from boltz2_client import convert_a3m_to_multimer_csv, create_paired_msa_per_chain

a3m_paths = {chain: os.getenv(f"MULTIMER_A3M_{chain}") for chain in chain_sequences}
msa_per_chain = {}
if any(a3m_paths.values()):
    if not all(a3m_paths.values()):
        raise ValueError("Set both MULTIMER_A3M_A and MULTIMER_A3M_B")
    conversion = convert_a3m_to_multimer_csv(
        a3m_files={chain: Path(path) for chain, path in a3m_paths.items()},
        pairing_strategy="greedy",
    )
    if conversion.query_sequences != chain_sequences:
        raise ValueError("A3M query sequences must match chain_sequences")
    msa_per_chain = create_paired_msa_per_chain(conversion)
    for chain, csv_text in conversion.csv_per_chain.items():
        (output_dir / f"chain_{chain}.csv").write_text(csv_text)
    print("Using paired alignments:", list(msa_per_chain))
else:
    print("No A3M files supplied: predicting without an MSA.")

## Predict and inspect interface confidence

The client saves all mmCIF structures and JSON scores. Pairwise chain scores
are returned as per-sample dictionaries. Confidence is a model output, not
experimental evidence of binding.

In [ ]:
request = PredictionRequest(polymers=[
    Polymer(id=chain, molecule_type="protein", sequence=sequence, msa=msa_per_chain.get(chain))
    for chain, sequence in chain_sequences.items()
])
result = await client.predict(request, output_dir=output_dir / "heterodimer")
print("Confidence:", result.confidence_scores)
print("Interface TM:", result.iptm_scores)
print("Pairwise interface TM:", result.pair_chains_iptm_scores)
print("Saved outputs:", (output_dir / "heterodimer").resolve())

In [ ]:
import py3Dmol

view = py3Dmol.view(width=800, height=450)
view.addModel(result.structures[0].structure, "cif")
view.setStyle({"chain": "A"}, {"cartoon": {"color": "cyan"}})
view.setStyle({"chain": "B"}, {"cartoon": {"color": "salmon"}})
view.zoomTo()
view.show()

## Specify two copies of one chain

Use the same sequence with two distinct IDs. This input-format example uses
two barnase chains and does not establish that barnase forms a homodimer.

In [ ]:
homodimer_request = PredictionRequest(polymers=[
    Polymer(id=chain, molecule_type="protein", sequence=chain_sequences["A"])
    for chain in ("A", "B")
])
homodimer_result = await client.predict(homodimer_request, output_dir=output_dir / "homodimer")
print("Two-copy interface TM:", homodimer_result.iptm_scores)